# 00 — Comparación de modelos y análisis de negocio

Este notebook reúne resultados ya obtenidos en los notebooks de entrenamiento y construye el análisis de valor del sistema. No entrena modelos ni depende de la ejecución de los demás notebooks: las métricas y matrices se registran manualmente después de validar cada experimento.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

## Resultados técnicos

La tabla se completa con valores fijos para que este notebook funcione como síntesis independiente. `cv_roc_auc` corresponde a validación cruzada agrupada sobre 2015–2016 y `validation_roc_auc` al holdout temporal de enero–agosto de 2017.

In [2]:
model_results = pd.DataFrame([
    {
        "experiment": "00_dummy_prior",
        "model": "DummyClassifier",
        "feature_set": "No predictive features",
        "cv_roc_auc": np.nan,
        "cv_std": np.nan,
        "validation_roc_auc": 0.5000,
        "accuracy": 0.6130,
        "precision": 0.0000,
        "recall": 0.0000,
        "f1": 0.0000,
        "threshold": 0.5,
    },
    {
        "experiment": "01_logistic_base",
        "model": "Logistic Regression",
        "feature_set": "Original cleaned",
        "cv_roc_auc": 0.8925,
        "cv_std": 0.0036,
        "validation_roc_auc": 0.8422,
        "accuracy": 0.7061,
        "precision": 0.5798,
        "recall": 0.8740,
        "f1": 0.6971,
        "threshold": 0.5,
    },
])
display(model_results)

,experiment,model,feature_set,cv_roc_auc,cv_std,validation_roc_auc,accuracy,precision,recall,f1,threshold
0,00_dummy_prior,DummyClassifier,No predictive features,NaN,NaN,0.5000,0.6130,0.0000,0.000,0.0000,0.5
1,01_logistic_base,Logistic Regression,Original cleaned,0.8925,0.0036,0.8422,0.7061,0.5798,0.874,0.6971,0.5


## Resultados de clasificación para el escenario de negocio

El análisis económico depende del umbral y, por tanto, de la matriz de confusión. Los conteos siguientes corresponden al holdout temporal y al umbral 0,5. Cuando se evalúen otros umbrales o modelos se agregará una nueva fila sin sobrescribir el experimento anterior.

In [3]:
classification_outcomes = pd.DataFrame([
    {
        "experiment": "01_logistic_base",
        "threshold": 0.5,
        "true_negatives": 14967,
        "false_positives": 9975,
        "false_negatives": 1984,
        "true_positives": 13761,
    },
])
display(classification_outcomes)

,experiment,threshold,true_negatives,false_positives,false_negatives,true_positives
0,01_logistic_base,0.5,14967,9975,1984,13761


## Resultados operativos de cancelaciones críticas

Se considera crítico un `No-Show` o una cancelación registrada entre cero y siete días antes de la llegada. El modelo continúa prediciendo cancelaciones generales; esta clasificación se utiliza únicamente para evaluar qué parte de los eventos más perjudiciales logra identificar.

El valor bruto estimado de una reserva se calcula como `ADR × TotalNights`. Se excluyen de la valoración el registro con `ADR = 5400` y el ADR negativo. Esta medida no equivale a una pérdida contable observada. Los valores se copian del bloque de evaluación operativa del notebook baseline después de ejecutarlo.

In [4]:
critical_cancellation_results = pd.DataFrame([
    {
        "experiment": "01_logistic_base",
        "late_cancellation_days": 7,
        "critical_events": 1598,
        "critical_events_detected": 1179,
        "critical_events_missed": 419,
        "interventions": 23736,
        "non_critical_interventions": 22557,
        "estimated_critical_value": 540098.62,
        "estimated_critical_value_detected": 419003.64,
        "estimated_critical_value_missed": 121094.98,
    },
])
critical_cancellation_results["critical_recall"] = (
    critical_cancellation_results["critical_events_detected"]
    / critical_cancellation_results["critical_events"]
)
critical_cancellation_results["critical_intervention_precision"] = (
    critical_cancellation_results["critical_events_detected"]
    / critical_cancellation_results["interventions"]
)
critical_cancellation_results["critical_value_capture"] = (
    critical_cancellation_results["estimated_critical_value_detected"]
    / critical_cancellation_results["estimated_critical_value"]
)
display(critical_cancellation_results.round(4))

,experiment,late_cancellation_days,critical_events,critical_events_detected,critical_events_missed,interventions,non_critical_interventions,estimated_critical_value,estimated_critical_value_detected,estimated_critical_value_missed,critical_recall,critical_intervention_precision,critical_value_capture
0,01_logistic_base,7,1598,1179,419,23736,22557,540098.62,419003.64,121094.98,0.7378,0.0497,0.7758


## Comparación con intervención aleatoria

Una referencia más informativa que no hacer nada consiste en seleccionar aleatoriamente la misma cantidad de reservas que marca el modelo. Bajo selección aleatoria, la proporción esperada de eventos críticos detectados coincide con la proporción intervenida del holdout. El lift compara la cobertura real del modelo con esa expectativa.

In [5]:
validation_rows = int(classification_outcomes[[
    "true_negatives", "false_positives", "false_negatives", "true_positives"
]].sum(axis=1).iloc[0])
critical_row = critical_cancellation_results.iloc[0]
intervention_rate = critical_row["interventions"] / validation_rows
random_expected_critical_events = critical_row["critical_events"] * intervention_rate
critical_capture_lift = (
    critical_row["critical_events_detected"] / random_expected_critical_events
)

operational_comparison = pd.DataFrame([
    {
        "strategy": "Random selection",
        "interventions": critical_row["interventions"],
        "intervention_rate": intervention_rate,
        "critical_events_detected": random_expected_critical_events,
        "critical_event_recall": intervention_rate,
        "lift_vs_random": 1.0,
    },
    {
        "strategy": "Logistic Regression (threshold 0.5)",
        "interventions": critical_row["interventions"],
        "intervention_rate": intervention_rate,
        "critical_events_detected": critical_row["critical_events_detected"],
        "critical_event_recall": critical_row["critical_recall"],
        "lift_vs_random": critical_capture_lift,
    },
])
display(operational_comparison.round(4))

,strategy,interventions,intervention_rate,critical_events_detected,critical_event_recall,lift_vs_random
0,Random selection,23736,0.5834,932.2419,0.5834,1.0000
1,Logistic Regression (threshold 0.5),23736,0.5834,1179.0000,0.7378,1.2647


## Modelo de valor esperado

El modelo predictivo no evita cancelaciones por sí mismo. Se supone que las reservas clasificadas como riesgosas reciben una acción preventiva con un costo y una efectividad determinada. Las funciones separan explícitamente:

- la pérdida media producida por una cancelación;
- el costo de aplicar la intervención;
- un posible costo adicional por intervenir innecesariamente sobre un falso positivo;
- y la proporción de cancelaciones detectadas que la acción consigue evitar.

También se incorpora una fracción no recuperada del valor bruto. Una fracción igual a uno representa el supuesto extremo de que una cancelación crítica pierde todo el valor y la habitación no puede revenderse. Los parámetros no deben presentarse como hechos observados: son supuestos para análisis de sensibilidad.

In [6]:
def validate_business_parameters(parameters):
    required = {
        "unrecovered_value_fraction",
        "cost_per_intervention",
        "false_positive_extra_cost",
        "intervention_effectiveness",
    }
    missing = sorted(required.difference(parameters))
    if missing:
        raise ValueError(f"Faltan parámetros de negocio: {missing}")
    if any(parameters[key] < 0 for key in required):
        raise ValueError("Los costos, fracciones y efectividad no pueden ser negativos.")
    for key in ["unrecovered_value_fraction", "intervention_effectiveness"]:
        if parameters[key] > 1:
            raise ValueError(f"{key} debe estar entre 0 y 1.")


def calculate_model_policy(critical_results, parameters):
    validate_business_parameters(parameters)
    interventions = float(critical_results["interventions"])
    non_critical_interventions = float(critical_results["non_critical_interventions"])
    total_critical_value = float(critical_results["estimated_critical_value"])
    detected_critical_value = float(critical_results["estimated_critical_value_detected"])
    missed_critical_value = float(critical_results["estimated_critical_value_missed"])

    unrecovered_fraction = parameters["unrecovered_value_fraction"]
    effectiveness = parameters["intervention_effectiveness"]
    no_action_cost = total_critical_value * unrecovered_fraction
    intervention_cost = interventions * parameters["cost_per_intervention"]
    false_positive_cost = (
        non_critical_interventions * parameters["false_positive_extra_cost"]
    )
    remaining_cancellation_cost = unrecovered_fraction * (
        missed_critical_value + detected_critical_value * (1 - effectiveness)
    )
    model_policy_cost = intervention_cost + false_positive_cost + remaining_cancellation_cost
    expected_savings = no_action_cost - model_policy_cost

    return {
        "no_action_cost": no_action_cost,
        "model_policy_cost": model_policy_cost,
        "expected_savings": expected_savings,
        "relative_savings": expected_savings / no_action_cost if no_action_cost else np.nan,
        "interventions": interventions,
        "expected_value_recovered": detected_critical_value * unrecovered_fraction * effectiveness,
        "interventions": interventions,
    }


def calculate_break_even_effectiveness(critical_results, parameters):
    validate_business_parameters(parameters)
    interventions = float(critical_results["interventions"])
    non_critical_interventions = float(critical_results["non_critical_interventions"])
    detected_value = float(critical_results["estimated_critical_value_detected"])
    avoidable_loss = detected_value * parameters["unrecovered_value_fraction"]
    if avoidable_loss == 0:
        return np.inf
    policy_cost_before_effect = (
        interventions * parameters["cost_per_intervention"]
        + non_critical_interventions * parameters["false_positive_extra_cost"]
    )
    return policy_cost_before_effect / avoidable_loss

## Escenario ilustrativo editable

Los parámetros siguientes representan el escenario máximo en el que no se recupera ninguna parte del valor de las cancelaciones críticas. Los costos y la efectividad de la intervención todavía son ilustrativos y deberán evaluarse mediante varios escenarios.

In [7]:
business_parameters = {
    "unrecovered_value_fraction": 1.0,
    "cost_per_intervention": 10.0,
    "false_positive_extra_cost": 0.0,
    "intervention_effectiveness": 0.25,
}

selected_critical_result = critical_cancellation_results.loc[
    critical_cancellation_results["experiment"].eq("01_logistic_base")
].iloc[0]

required_result_columns = [
    "non_critical_interventions", "estimated_critical_value",
    "estimated_critical_value_detected", "estimated_critical_value_missed",
]
if selected_critical_result[required_result_columns].isna().any():
    print("Análisis pendiente: ejecutar el bloque operativo del baseline y copiar sus resultados.")
else:
    business_result = calculate_model_policy(selected_critical_result, business_parameters)
    business_result["break_even_effectiveness"] = calculate_break_even_effectiveness(
        selected_critical_result, business_parameters
    )
    display(pd.DataFrame([business_result]).round(4))

,no_action_cost,model_policy_cost,expected_savings,relative_savings,interventions,expected_value_recovered,break_even_effectiveness
0,540098.62,672707.71,-132609.09,-0.2455,23736.0,104750.91,0.5665


## Extensiones pendientes

El escenario basado en una única matriz permite verificar la lógica, pero el análisis final deberá trabajar con probabilidades individuales para:

- comparar diferentes umbrales;
- limitar la cantidad de intervenciones según la capacidad del hotel;
- comparar la priorización del modelo con una selección aleatoria;
- evaluar varios costos y niveles de efectividad;
- y representar la incertidumbre de los supuestos mediante análisis de sensibilidad.

La política sólo se considerará conveniente en aquellos escenarios donde produzca ahorro esperado positivo y mantenga una cantidad de intervenciones operativamente viable.

## Conclusión provisional del baseline

La regresión logística base contiene señal útil: supera la selección aleatoria con igual cantidad de intervenciones y captura una proporción importante de los eventos críticos y de su valor bruto estimado. Sin embargo, con umbral 0,5 requiere intervenir sobre el 58,34 % del holdout y sólo el 4,97 % de esas intervenciones corresponde a un evento crítico. Bajo los parámetros económicos ilustrativos actuales, esta política no resulta conveniente.

Esta conclusión se limita al modelo baseline, al umbral y a los supuestos analizados. No demuestra todavía que todo uso de modelos predictivos carezca de valor: Feature Engineering, otros algoritmos y una política de capacidad limitada podrían concentrar mejor las intervenciones.

## Decisiones posteriores al baseline

El análisis de redundancia entre predictores no mostró multicolinealidad preocupante dentro del bloque numérico: el mayor VIF fue 1,52. Sin embargo, `ArrivalDateMonth` y `ArrivalDateWeekNumber` presentan una asociación casi determinística ($\eta=0{,}9951$). Para los conjuntos mejorados se elimina `ArrivalDateWeekNumber` y se conservan mes, año y día. El baseline histórico mantiene sus 25 features originales para preservar la trazabilidad de sus resultados.